# Notebook 3 — Modello LSTM

Obiettivo: addestrare e valutare l'LSTM su tutti e 4 i sotto-dataset CMAPSS.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from src.models.lstm_model import build_lstm
from src.training import run_experiment
from src.metrics import evaluate

DATA_DIR = '../data/processed'
MODEL_DIR = '../saved_models'

## Architettura LSTM

```
Input (30, 17)
    │
LSTM(128, return_sequences=True)   ← primo layer: output per ogni timestep
    │
Dropout(0.2)
    │
LSTM(64)                           ← secondo layer: distilla la rappresentazione
    │
Dropout(0.2)
    │
Dense(64, relu)
    │
Dense(1, linear)                   ← predizione RUL (regressione)
```

**Scelte architetturali:**
- **Stacked LSTM**: il primo layer cattura pattern locali, il secondo integra le dipendenze a lungo termine
- **return_sequences=True** nel primo LSTM: passa la sequenza completa al secondo layer
- **units dimezzati** nel secondo LSTM (128 → 64): gerarchia di astrazione, riduce la dimensione
- **Loss MSE**: penalizza gli errori grandi quadraticamente, coerente con RMSE come metrica

**Callbacks di training:**
- `EarlyStopping(patience=15)`: ferma se val_loss non migliora → spiega il diverso numero di epoche tra dataset
- `ModelCheckpoint`: salva solo il miglior modello, non l'ultimo
- `ReduceLROnPlateau`: dimezza il lr dopo 7 epoche stagnanti → fine-tuning automatico

In [ ]:
results = {}
for fd in ['FD001', 'FD002', 'FD003', 'FD004']:
    # Carico gli array preprocessati salvati dal notebook 02
    X_train = np.load(f'{DATA_DIR}/X_train_{fd}.npy')
    y_train = np.load(f'{DATA_DIR}/y_train_{fd}.npy')
    X_test  = np.load(f'{DATA_DIR}/X_test_{fd}.npy')
    y_test  = np.load(f'{DATA_DIR}/y_test_{fd}.npy')

    # window_size e n_features estratti dinamicamente → il modello si adatta ai dati
    window_size = X_train.shape[1]  # 30
    n_features  = X_train.shape[2]  # 17

    # Costruisco il modello: units=128 (primo LSTM), dropout=0.2
    model = build_lstm(window_size, n_features, units=128, dropout=0.2)
    model_path = f'{MODEL_DIR}/lstm_{fd}.keras'

    # run_experiment: train → salva best checkpoint → carica best model → valuta su test
    res = run_experiment(model, X_train, y_train, X_test, y_test, model_path)
    results[fd] = res
    print(f'{fd}: {res["metrics"]}')

## Analisi delle curve di training

Le curve LSTM mostrano una struttura caratteristica **a gradino**:
1. Discesa rapida nelle prime epoche (apprendimento delle statistiche di base)
2. Plateau intermedio (la rete è "bloccata" in un minimo locale)
3. Seconda discesa brusca quando `ReduceLROnPlateau` abbassa il learning rate

**Train ≈ Val** in tutti i dataset: nessun overfitting, la regularizzazione (dropout + early stopping) funziona.

Il numero di epoche varia tra dataset:
- FD001/FD003 (~85-90 epoche): dataset più piccoli, convergenza più lenta
- FD002/FD004 (~45-75 epoche): più campioni per batch → aggiornamenti più stabili, convergenza più rapida

In [1]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (fd, res) in zip(axes.flatten(), results.items()):
    ax.plot(res['history']['loss'], label='train')
    ax.plot(res['history']['val_loss'], label='val')
    ax.set_title(f'LSTM — {fd}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    # Se train e val si sovrappongono → no overfitting (buon segnale)
    ax.legend()
plt.tight_layout()
plt.savefig('../plots/03_lstm_training_curves.png', dpi=150)
plt.show()

NameError: name 'plt' is not defined